In [ ]:
import os
import sys
os.chdir('D:/CEOS') 
sys.path.append(os.path.abspath('D:/CEOS/Modules'))
[sys.path.append(x[0]) for x in os.walk('.')]

import pandas as pd
import toolsv5 as tools
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import datetime 
import statsmodels.api as sm
from scipy import stats
import pickle 

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
pd.set_option('display.max_rows', None)

# Read in experiment results

In [ ]:
with open('D:/CEOS/InputData/groundedeohls/HLS_SiteUni_LAI.pkl', 'rb') as file:
    HLS_SiteUni_LAI = pickle.load(file)

with open('D:/CEOS/InputData/groundedeohls/HLS_SiteAll_LAI.pkl', 'rb') as file:
    HLS_SiteAll_LAI = pickle.load(file)

with open('D:/CEOS/InputData/groundedeohls/HLS_SiteAllWide_LAI.pkl', 'rb') as file:
    HLS_SiteAllWide_LAI = pickle.load(file)
    
with open('D:/CEOS/InputData/groundedeohls/HLS_All_LAI.pkl', 'rb') as file:
    HLS_All_LAI = pickle.load(file)

with open('D:/CEOS/InputData/groundedeohls/HLS_AllWide_LAI.pkl', 'rb') as file:
    HLS_AllWide_LAI = pickle.load(file)

with open('D:/CEOS/InputData/groundedeohls/S2_AllWide_LAI.pkl', 'rb') as file:
    S2_AllWide_LAI = pickle.load(file)

with open('D:/CEOS/InputData/groundedeohls/HLS_Stage4Wide_LAI.pkl', 'rb') as file:
    HLS_Stage4Wide_LAI = pickle.load( file)


In [ ]:
with open('D:/CEOS/InputData/groundedeohls/HLS_SiteUni_FAPAR.pkl', 'rb') as file:
    HLS_SiteUni_FAPAR = pickle.load(file)

with open('D:/CEOS/InputData/groundedeohls/HLS_SiteAll_FAPAR.pkl', 'rb') as file:
    HLS_SiteAll_FAPAR = pickle.load(file)


with open('D:/CEOS/InputData/groundedeohls/HLS_SiteAllWide_FAPAR.pkl', 'rb') as file:
    HLS_SiteAllWide_FAPAR = pickle.load(file)
    
with open('D:/CEOS/InputData/groundedeohls/HLS_All_FAPAR.pkl', 'rb') as file:
    HLS_All_FAPAR = pickle.load(file)

with open('D:/CEOS/InputData/groundedeohls/HLS_AllWide_FAPAR.pkl', 'rb') as file:
    HLS_AllWide_FAPAR = pickle.load(file)

with open('D:/CEOS/InputData/groundedeohls/S2_AllWide_FAPAR.pkl', 'rb') as file:
    S2_AllWide_FAPAR = pickle.load(file)

with open('D:/CEOS/InputData/groundedeohls/HLS_Stage4Wide_FAPAR.pkl', 'rb') as file:
    HLS_Stage4Wide_FAPAR = pickle.load(file)

In [ ]:
print(HLS_SiteAllWide_LAI['data'].head(1)['validation_stats'])
print(HLS_AllWide_LAI['data'].head(1)['validation_stats'])

In [ ]:

HLS_All_LAI['name'] = 'HLS_All_LAI'
validation_dict_list = [HLS_SiteAllWide_LAI,HLS_AllWide_LAI]
combined_df = pd.DataFrame()
for validation_dict in validation_dict_list:
    df1 = validation_dict['data']
    variable = validation_dict['variable']
    #augment df with standard quatities
    df1 = df1.assign(y = df1[variable+'_FRM'].to_numpy(),
                 x = df1['medianestimate'+variable].to_numpy(),
                 x_ci = df1['medianestimate'+variable+'_ci'].to_numpy(),
                 r = df1['medianresidual'+variable].to_numpy(),
                r_ci = np.sqrt(np.power(df1['medianestimate'+variable+'_ci'].to_numpy(),2)+np.power(df1[variable+'_ci'].to_numpy(),2)),
                A = df1['validation_stats'].str.get('A').str.get('q').to_numpy(),     
                A_lower = 1.35*df1['validation_stats'].str.get('A').str.get('ci_lower').to_numpy(),
                A_upper = 1.35*df1['validation_stats'].str.get('A').str.get('ci_lower').to_numpy(),
                A_ci = 1.35*df1['validation_stats'].str.get('A').str.get('ci').to_numpy(),
                A_N = df1['validation_stats'].str.get('A').str.get('N').to_numpy(),     
                B = df1['validation_stats'].str.get('B').str.get('q').to_numpy(),     
                B_lower = 1.35*df1['validation_stats'].str.get('B').str.get('ci_lower').to_numpy(),
                B_upper = 1.35*df1['validation_stats'].str.get('B').str.get('ci_lower').to_numpy(),
                B_ci = 1.35*df1['validation_stats'].str.get('B').str.get('ci').to_numpy(),
                B_N = df1['validation_stats'].str.get('B').str.get('N').to_numpy(),     
                U = df1['validation_stats'].str.get('U').str.get('q').to_numpy(),     
                U_lower = 1.35*df1['validation_stats'].str.get('U').str.get('ci_lower').to_numpy(),
                U_upper = 1.35*df1['validation_stats'].str.get('U').str.get('ci_lower').to_numpy(),
                U_ci = 1.35*df1['validation_stats'].str.get('U').str.get('ci').to_numpy(),
                U_N = df1['validation_stats'].str.get('U').str.get('N').to_numpy(),     
                S = df1['validation_stats'].str.get('S').str.get('slope').to_numpy(),     
                S_lower = df1['validation_stats'].str.get('S').str.get('upper_ci').to_numpy(),
                S_upper = df1['validation_stats'].str.get('S').str.get('lower_ci').to_numpy(),
                S_ci = df1['validation_stats'].str.get('S').str.get('ci').to_numpy(),
                S_N = df1['validation_stats'].str.get('S').str.get('N').to_numpy()     
                )
    df1['name']= validation_dict['name']
    #drop zero uncertainty estimates that imply FRM uncertainty was too large
    df1 = df1[df1['U']>0]
    combined_df = pd.concat([combined_df,df1],axis=0, ignore_index=True)
fig,ax=plt.subplots(1,1,figsize=(10,10),layout='constrained')
ax=sns.histplot(combined_df,x='U_N',hue='name',alpha=0.6)
ax.set_xlabel('$\hat{n}$ Uncertainty LAI')
print(combined_df.groupby(['name','NLCD_group'])['U_N'].count())
print(combined_df[combined_df['U_N']>10].groupby(['name','NLCD_group'])['U_N'].count())
print(combined_df.groupby(['name','NLCD_group'])['S_N'].count())
print(combined_df[combined_df['S_N']>10].groupby(['name','NLCD_group'])['S_N'].count())

In [ ]:

HLS_All_FAPAR['name'] = 'HLS_All_FAPAR'
validation_dict_list = [HLS_SiteAllWide_FAPAR,HLS_AllWide_FAPAR]
combined_df = pd.DataFrame()
for validation_dict in validation_dict_list:
    df1 = validation_dict['data']
    variable = validation_dict['variable']
    #augment df with standard quatities
    df1 = df1.assign(y = df1[variable+'_FRM'].to_numpy(),
                 x = df1['medianestimate'+variable].to_numpy(),
                 x_ci = df1['medianestimate'+variable+'_ci'].to_numpy(),
                 r = df1['medianresidual'+variable].to_numpy(),
                r_ci = np.sqrt(np.power(df1['medianestimate'+variable+'_ci'].to_numpy(),2)+np.power(df1[variable+'_ci'].to_numpy(),2)),
                A = df1['validation_stats'].str.get('A').str.get('q').to_numpy(),     
                A_lower = 1.35*df1['validation_stats'].str.get('A').str.get('ci_lower').to_numpy(),
                A_upper = 1.35*df1['validation_stats'].str.get('A').str.get('ci_lower').to_numpy(),
                A_ci = 1.35*df1['validation_stats'].str.get('A').str.get('ci').to_numpy(),
                A_N = df1['validation_stats'].str.get('A').str.get('N').to_numpy(),     
                B = df1['validation_stats'].str.get('B').str.get('q').to_numpy(),     
                B_lower = 1.35*df1['validation_stats'].str.get('B').str.get('ci_lower').to_numpy(),
                B_upper = 1.35*df1['validation_stats'].str.get('B').str.get('ci_lower').to_numpy(),
                B_ci = 1.35*df1['validation_stats'].str.get('B').str.get('ci').to_numpy(),
                B_N = df1['validation_stats'].str.get('B').str.get('N').to_numpy(),     
                U = df1['validation_stats'].str.get('U').str.get('q').to_numpy(),     
                U_lower = 1.35*df1['validation_stats'].str.get('U').str.get('ci_lower').to_numpy(),
                U_upper = 1.35*df1['validation_stats'].str.get('U').str.get('ci_lower').to_numpy(),
                U_ci = 1.35*df1['validation_stats'].str.get('U').str.get('ci').to_numpy(),
                U_N = df1['validation_stats'].str.get('U').str.get('N').to_numpy(),     
                S = df1['validation_stats'].str.get('S').str.get('slope').to_numpy(),     
                S_lower = df1['validation_stats'].str.get('S').str.get('upper_ci').to_numpy(),
                S_upper = df1['validation_stats'].str.get('S').str.get('lower_ci').to_numpy(),
                S_ci = df1['validation_stats'].str.get('S').str.get('ci').to_numpy(),
                S_N = df1['validation_stats'].str.get('S').str.get('N').to_numpy()     
                )
    df1['name']= validation_dict['name']
    #drop zero uncertainty estimates that imply FRM uncertainty was too large
    df1 = df1[df1['U']>0]
    combined_df = pd.concat([combined_df,df1],axis=0, ignore_index=True)
fig,ax=plt.subplots(1,1,figsize=(10,10),layout='constrained')
ax=sns.histplot(combined_df,x='U_N',hue='name',alpha=0.6)
ax.set_xlabel('$\hat{n}$ Uncertainty FAPAR')
plt.show()

print(combined_df.groupby(['name','NLCD_group'])['U_N'].count())
print(combined_df[combined_df['U_N']>10].groupby(['name','NLCD_group'])['U_N'].count())
print(combined_df.groupby(['name','NLCD_group'])['S_N'].count())
print(combined_df[combined_df['S_N']>10].groupby(['name','NLCD_group'])['S_N'].count())

# Figure 1

In [ ]:

tools.visualize_validation([HLS_SiteUni_LAI,HLS_SiteAll_LAI],variable='LAI',groups=['needleleafForest','broadleafForest','nonForest'],user_requirements={'Uabs':0.5,'Urel':0.15,'Sabs':0.06},minimum_df=10,bivariate=True,legend=True)

In [ ]:
HLS_SiteAll_FAPAR['name'] = 'HLS-SiteAll'
tools.visualize_validation([HLS_SiteUni_FAPAR,HLS_SiteAll_FAPAR],variable='FAPAR',groups=['needleleafForest','broadleafForest','nonForest'],user_requirements={'Uabs':0.005,'Urel':0.10,'Sabs':0.03},minimum_df=10,bivariate=True,legend=True)

In [ ]:
tools.visualize_validation([HLS_SiteAll_LAI,HLS_SiteAllWide_LAI],variable='LAI',groups=['needleleafForest','broadleafForest','nonForest'],user_requirements={'Uabs':0.5,'Urel':0.15,'Sabs':0.06},minimum_df=10,bivariate=True,legend=True)

In [ ]:
tools.visualize_validation([HLS_SiteAll_FAPAR,HLS_SiteAllWide_FAPAR],variable='FAPAR',groups=['needleleafForest','broadleafForest','nonForest'],user_requirements={'Uabs':0.005,'Urel':0.10,'Sabs':0.03},minimum_df=10,bivariate=True,legend=True)

In [ ]:
tools.visualize_validation([HLS_SiteAllWide_LAI,HLS_AllWide_LAI],variable='LAI',groups=['needleleafForest','broadleafForest','nonForest'],user_requirements={'Uabs':0.5,'Urel':0.15,'Sabs':0.06},minimum_df=10,bivariate=True,legend=True)

In [ ]:
tools.visualize_validation([S2_AllWide_LAI,HLS_AllWide_LAI],variable='LAI',groups=['needleleafForest','broadleafForest','nonForest'],user_requirements={'Uabs':0.5,'Urel':0.15,'Sabs':0.06},minimum_df=10,bivariate=True,legend=True)

In [ ]:
tools.visualize_validation([S2_AllWide_FAPAR,HLS_AllWide_FAPAR],variable='FAPAR',groups=['needleleafForest','broadleafForest','nonForest'],user_requirements={'Uabs':0.005,'Urel':0.10,'Sabs':0.03},minimum_df=10,bivariate=True,legend=True)

In [ ]:
tools.visualize_validation([HLS_SiteAllWide_FAPAR,HLS_AllWide_FAPAR],variable='FAPAR',groups=['needleleafForest','broadleafForest','nonForest'],user_requirements={'Uabs':0.005,'Urel':0.10,'Sabs':0.03},minimum_df=10,bivariate=True,legend=True)

In [ ]:
tools.visualize_validation([S2_AllWide_FAPAR,HLS_AllWide_FAPAR],variable='FAPAR',groups=['needleleafForest','broadleafForest','nonForest'],user_requirements={'Uabs':0.005,'Urel':0.10,'Sabs':0.03},minimum_df=10,bivariate=True,legend=True)

In [ ]:
tools.visualize_validation([HLS_SiteUni_LAI,HLS_Stage4Wide_LAI],variable='LAI',groups=['needleleafForest','broadleafForest','nonForest'],user_requirements={'Uabs':0.5,'Urel':0.15,'Sabs':0.06},legend=True)

In [ ]:

tools.visualize_validation([S2_AllWide_LAI,HLS_AllWide_LAI],variable='LAI',groups=['needleleafForest','broadleafForest','nonForest'],user_requirements={'Uabs':0.5,'Urel':0.15,'Sabs':0.06},legend=True)

In [ ]:
tools.visualize_validation([HLS_SiteUni_FAPAR,HLS_AllWide_FAPAR],variable='FAPAR',groups=['needleleafForest','broadleafForest','nonForest'],user_requirements={'Uabs':0.005,'Urel':0.10,'Sabs':0.03},legend=True)

In [ ]:
tools.visualize_validation([S2_AllWide_FAPAR,HLS_AllWide_FAPAR],variable='FAPAR',groups=['needleleafForest','broadleafForest','nonForest'],user_requirements={'Uabs':0.005,'Urel':0.10,'Sabs':0.03},legend=True)